**Proyecto II de Programación para Estadística ll**

**Tema:** La evolución del gasto corriente en salud (% del PIB), la pobreza y la incidencia de las enfermedades transmisibles y su variación entre países con diferentes desarrollos económicos durante la última década

**Profesor:** Michael Sanchez Soto

**Estudiantes:** Cyra Coto, Gustavo Jiménez y Abigail Murillo

**Pregunta**

¿Cómo ha evolucionado el gasto corriente en salud (% del PIB), la pobreza y la incidencia de las enfermedades transmisibles durante la última década y de qué manera varía esta relación entre países con diferentes niveles de desarrollo económico?

# 1. CARGA DE LAS BASES DE DATOS

In [4]:
#Importación de las librerías y carga de los datas frames crudos
import pandas as pd
import numpy as np
import json
from google.colab import files

# -----------------------
# 1. CARGA DE ARCHIVOS RAW
# -----------------------

print("SUBE ARCHIVO DENGUE 1")
uploaded_dengue = files.upload()
df_dengue = pd.read_excel(list(uploaded_dengue.keys())[0])

print("SUBE ARCHIVO HIV")
uploaded_hiv = files.upload()
df_HIV = pd.read_excel(
    list(uploaded_hiv.keys())[0],
    sheet_name="HIV-Test-&-Treat_ByYear",
    engine="openpyxl",
    skiprows=6
)

print("SUBE ARCHIVO INDICE DE POBREZA")
uploaded_pobreza = files.upload()
df_indice_pobreza = pd.read_csv(list(uploaded_pobreza.keys())[0])

print("SUBE ARCHIVO TUBERCULOSIS")
uploaded_tb = files.upload()
df_tuberculosis = pd.read_csv(list(uploaded_tb.keys())[0])

# -----------------------
# 2. CARGA DEL JSON
# -----------------------

print("SUBE ARCHIVO JSON DE GASTO EN SALUD")
uploaded_json = files.upload()
json_filename = list(uploaded_json.keys())[0]
print("JSON cargado:", json_filename)

# Leer JSON correctamente
with open(json_filename, "r", encoding="utf-8") as f:
    data_json = json.load(f)

# Normalizar JSON en DataFrame
df_gasto_salud = pd.json_normalize(data_json)

# -----------------------
# 3. SEGUNDO DATA FRAME DEL DENGUE
# -----------------------

print("SUBE ARCHIVO DENGUE 2")
uploaded_dengue2 = files.upload()
df_dengue2 = pd.read_csv(list(uploaded_dengue2.keys())[0])


SUBE ARCHIVO DENGUE 1


Saving indice de pobreza global.csv to indice de pobreza global (1).csv


ValueError: Excel file format cannot be determined, you must specify an engine manually.

# 2. PREPARACIÓN DE LA BASE DEL DENGUE

In [ ]:
df_dengue.columns #exploracion de las columnas del data frame del dengue

In [ ]:
display(df_dengue) #exploracion del df crudo

In [ ]:
df_dengue.isna().sum() #La base tiene varios varios NA's

In [ ]:
df_dengue.dtypes #Vemos que que las variables están en el formato correcto, además todos los valores faltantes son númericos, por lo que tenemos la posibilidad de sustituirlos con el valor de la mediana


In [ ]:
df_dengue["cases"] = df_dengue["cases"].fillna("No indica")
df_dengue["confirmed_cases"] = df_dengue["confirmed_cases"].fillna("No indica") #LLenamos los valores faltantes con 0 ya que no hay registros
df_dengue["severe_cases"] = df_dengue["severe_cases"].fillna("No indica")
df_dengue["deaths"] = df_dengue["deaths"].fillna("No indica")
df_dengue["cfr"] = df_dengue["cfr"].fillna("Desconocido")
df_dengue["cfr_ci_upper"] = df_dengue["cfr_ci_upper"].fillna("Desconocido") #No sabemos hasta que número podría llegar
df_dengue["cfr_ci_lower"] = df_dengue["cfr_ci_lower"].fillna("Desconocido")
df_dengue["prop_sev"] = df_dengue["prop_sev"].fillna("Desconocido")
df_dengue["prop_sev_ci_lower"] = df_dengue["prop_sev_ci_lower"].fillna("Desconocido")
df_dengue["prop_sev_ci_upper"] = df_dengue["prop_sev_ci_upper"].fillna("Desconocido") #No sabemos hasta que número podría llegar
df_dengue["prop_sev_ci_lower"] = df_dengue["prop_sev_ci_lower"].fillna("Desconocido")



In [ ]:
df_dengue.isna().sum() # Hemos limpiado los NA's

In [ ]:
df_dengue.duplicated().sum() #No posee duplicados

In [ ]:

#Asegúrate de que 'date' sea datetime
df_dengue['date'] = pd.to_datetime(df_dengue['date'])

# Crear columna 'year'
df_dengue['year'] = df_dengue['date'].dt.year


# Convertir 'cases' a numérico, forzando errores a NaN
df_dengue['cases'] = pd.to_numeric(df_dengue['cases'], errors='coerce')

# Agrupar por país y año
df_dengue_final = df_dengue.groupby(['country', 'year'], as_index=False)['cases'].sum()

df_dengue_final.to_excel("dengue.xlsx", index=False)


## 2.2 SEGUNDA BASE DEL DENGUE

In [ ]:
#quedarnos solo con columnas importantes ---
df_dengue2 = df_dengue2[["adm_0_name", "ISO_A0", "Year", "dengue_total"]]

#convertir 'dengue_total' a numérico por si viene como string ---
df_dengue2["dengue_total"] = pd.to_numeric(df_dengue2["dengue_total"], errors="coerce")

#agrupar por país y año ---
df_dengue2_final = (
    df_dengue2.groupby(["adm_0_name", "ISO_A0", "Year"], as_index=False)
      .agg(total_cases=("dengue_total", "sum"))
)

#renombrar

df_dengue2_final.rename(columns={
    'adm_0_name':'country',
    'ISO_A0':'codigo',
    'Year':'year',
    'total_cases':'dengue_cases'
},inplace=True)

#Resultado final
df_dengue2_final.head(10)

# 3. PREPARACION DE LA BASE DE LA TUBERCULOSIS

In [ ]:
display(df_tuberculosis)

In [ ]:
df_tuberculosis.isna().sum() #La base posee varios NA's

In [ ]:
df_tuberculosis.dtypes # Todas las variables son del tipo correcto de datos

In [ ]:
df_tuberculosis["Code"] = df_tuberculosis["Code"].fillna("No especifica")

# 4. PREPARACION DE LA BASE GASTOS EN SALUD json

In [ ]:
#Esta base es la de tipo json. requiere un tratamiento especial
#Quitar las primeras filas que no aportan informacion
df_temp = df_gasto_salud.copy()
df_temp = df_temp.iloc[2:].reset_index(drop=True)

#Usar la primera fila como encabezado real
#La nueva fila 0 ahora tiene los nombres reales de columnas (Country Name, Country Code, Indicator Name, 2000, 2001, ...)
new_header = df_temp.iloc[0]
df_temp = df_temp.iloc[1:].reset_index(drop=True)
df_temp.columns = new_header

#Eliminar columnas 'Unnamed' y vacías que hay varias
df_temp = df_temp.loc[:, ~df_temp.columns.astype(str).str.contains(r'^Unnamed', na=False)]
df_temp = df_temp[[c for c in df_temp.columns if str(c).strip() != '' and c is not None]]

#Pasar a formato largo (tidy) year–value o hacer la transposición
#nombres de columnas
rename_map = {
    'Country Name': 'country_name',
    'Country Code': 'country_code',
    'Indicator Name': 'indicator_name',
    'Indicator Code': 'indicator_code'  # por si existiera
}
df_temp = df_temp.rename(columns={k: v for k, v in rename_map.items() if k in df_temp.columns})

#Detectar columnas de año: están como '2000.0', '2001.0',...
def es_col_anio(col):
    s = str(col).strip()
    s = s[:-2] if s.endswith('.0') else s
    return s.isdigit() and len(s) == 4

year_cols = [c for c in df_temp.columns if es_col_anio(c)]
if not year_cols:
    raise ValueError("No se detectaron columnas de año. Revisa los nombres (p.ej. '2000.0').")

#Construir id_vars presentes
id_vars = [c for c in ['country_name', 'country_code', 'indicator_name', 'indicator_code'] if c in df_temp.columns]

df_largo = df_temp.melt(
    id_vars=id_vars,
    value_vars=year_cols,
    var_name='year_raw',
    value_name='value'
)

#Convertir 'year_raw' a 'year' numérico limpio
def limpiar_year(y):
    s = str(y).strip()
    s = s[:-2] if s.endswith('.0') else s
    return pd.to_numeric(s, errors='coerce')

df_largo['year'] = df_largo['year_raw'].apply(limpiar_year)
df_largo.drop(columns=['year_raw'], inplace=True)

#Tipos y limpieza básica
df_largo['value'] = pd.to_numeric(df_largo['value'], errors='coerce')
df_largo = df_largo.dropna(subset=['year', 'value'])

#Quitar filas sin país o con 'None'
if 'country_name' in df_largo.columns:
    df_largo = df_largo[df_largo['country_name'].notna() & (df_largo['country_name'].astype(str).str.strip().str.lower() != 'none')]

#Filtrar el indicador que interesa
target_names = [
    "Gasto corriente en salud (% del PIB)",
    "Current health expenditure (% of GDP)"
]
df_temporal2 = df_largo[df_largo['indicator_name'].isin(target_names)].copy()

#dejar sólo países con código ISO3 (evitar agregados/regiones)
if 'country_code' in df_temporal2.columns:
    df_temporal2 = df_temporal2[df_temporal2['country_code'].astype(str).str.len() == 3]

#Orden final
df_temporal2 = df_temporal2.sort_values(by=['country_name', 'year']).reset_index(drop=True)

out_min = df_temporal2.rename(columns={'country_name': 'pais'})[['pais', 'year', 'value']]

#metadatos útiles para merge:
out_full = df_temporal2.rename(columns={'country_name': 'pais'})[['pais', 'country_code', 'indicator_name', 'year', 'value']]

# Reordenar columnas: country_name, country_code, indicator_name, year, value
cols_order = ['country_name', 'country_code', 'indicator_name', 'year', 'value']
df_gastos_en_salud_final = df_temporal2[cols_order].copy()

#asegurar tipos
df_gastos_en_salud_final['year'] = pd.to_numeric(df_gastos_en_salud_final['year'], errors='coerce').astype('Int64')
df_gastos_en_salud_final['value'] = pd.to_numeric(df_gastos_en_salud_final['value'], errors='coerce')

#Ordenar filas por país, indicador y año
df_gastos_en_salud_final = df_gastos_en_salud_final.sort_values(by=['country_name', 'indicator_name', 'year']).reset_index(drop=True)

# Vista rápida
display(df_gastos_en_salud_final.head(100))



In [ ]:
#Ver si el dataframe tiene valores nulos
df_gastos_en_salud_final.isna().any()


# 5. PREPARACION DE LA BASE INDICE DE POBREZA

In [ ]:
display(df_indice_pobreza)
df_indice_pobreza

In [ ]:
df_indice_pobreza.isna().sum() #La base posee varios NA's

In [ ]:
df_indice_pobreza["ppp"]

In [ ]:
df_indice_pobreza["ppp"].max()

In [ ]:
df_indice_pobreza["ppp"].min()

In [ ]:
mediana_ppp = df_indice_pobreza["ppp"].median() # Utilizamos el valor de la mediana para rellenar valores faltantes

In [ ]:
df_indice_pobreza["watts"] # Watts es una de las columnas que posee NA's
df_indice_pobreza["watts"].max()

In [ ]:
df_indice_pobreza["watts"].min()

In [ ]:
mediana_watts =  df_indice_pobreza["watts"].median()
print(mediana_watts) # Utilizaremos la mediana para estimar rellenar los valores faltantes

In [ ]:
df_indice_pobreza["reporting_pce"] #reporting_pce posee NA's

In [ ]:
df_indice_pobreza["reporting_pce"].max()

In [ ]:
df_indice_pobreza["reporting_pce"].min()

In [ ]:
mediana_pce = df_indice_pobreza["reporting_pce"].median() #Utilizaremos la mediana para estimar los valores faltantes

In [ ]:
df_indice_pobreza["watts"] = df_indice_pobreza["watts"].fillna(df_indice_pobreza["watts"].median())
df_indice_pobreza["ppp"] = df_indice_pobreza["ppp"].fillna(df_indice_pobreza["ppp"].median())
df_indice_pobreza["reporting_pce"] = df_indice_pobreza["reporting_pce"].fillna(df_indice_pobreza["reporting_pce"].median())



In [ ]:
# Por otra parte, la columna "estimate_type" está completamente llena de NA's, por lo que no aporta nada de valor al análisis y es mejor eliminarla completamente.
print(df_indice_pobreza.columns)
df_indice_pobreza
df_indice_pobreza= df_indice_pobreza.drop(columns=["estimation_type"])


In [ ]:
df_indice_pobreza.isna().sum() # Ahora verificamos que el data frame no posee NA's

In [ ]:
df_indice_pobreza.dtypes #Los variables son del tipo que se espera de ellas

In [ ]:
df_indice_pobreza.duplicated().sum() # No tenemos columnas duplicadas

In [ ]:
df_indice_pobreza.head(10)

# 6. Preparacion de la base HIV

In [ ]:
df_HIV_subset = df_HIV.iloc[:, [0, 1, 2] + list(range(78, 83))]  #del data frame solo conservar estas columnas

df_HIV_subset.columns =['Anio', 'codigo', 'pais', 'todas_edades','ninos', 'mujeres', 'hombres', 'adultos'] #asignar un nombre


# Verificar
print(df_HIV_subset.head(5))
print(df_HIV_subset.dtypes)
#df_HIV_subset=df_HIV_subset.apply(pd.to_numeric, errors='coerce') #convertir a tipo numerico
#Verificar por NAs


# Lista de columnas que quieres convertir
cols_numericas = ['todas_edades', 'ninos', 'mujeres', 'hombres', 'adultos']

# Convertir esas columnas a numéricas (coerce convierte errores en NaN)
df_HIV_subset[cols_numericas] = df_HIV_subset[cols_numericas].apply(pd.to_numeric, errors='coerce')

print(df_HIV_subset.dtypes)
df_HIV_subset.isna().sum()
#al ser datos epidemiologicos no manipular NAs


In [ ]:
#PRUEBA DE FUNCIONAIENTO DE LA BASE
df_cr = df_HIV_subset[df_HIV_subset['codigo'] == 'CRI']
df_cr.head(5)


# Agrupar por año y sumar los casos (o cualquier otra agregación)
df_cr_grouped = df_cr.groupby('Anio')[['todas_edades', 'ninos', 'mujeres', 'hombres', 'adultos']].sum().reset_index()

# Verificar
print(df_cr_grouped)


# 7 UNIFICACIÓN DE LAS BASES merge

In [ ]:
#merge de las enfermedades


#Renombrar columnas para consistencia
df_tuberculosis.rename(columns={
    'Entity': 'country',
    'Year': 'year',
    'Estimated number of new cases of all forms of tuberculosis': 'tb_cases'
}, inplace=True)

df_HIV_subset.rename(columns={
    'Anio': 'year',
    'pais': 'country',
    'todas_edades': 'hiv_cases'
}, inplace=True)


#Normalizar nombres de país a mayúsculas (o minúsculas)
df_dengue2_final['country'] = df_dengue2_final['country'].str.upper()
df_tuberculosis['country'] = df_tuberculosis['country'].str.upper()
df_HIV_subset['country'] = df_HIV_subset['country'].str.upper()



#Merge progresivo por country y year
df_merge = df_dengue2_final.merge(df_tuberculosis[['country', 'year', 'tb_cases']],
                                 on=['country', 'year'], how='outer')

df_merge = df_merge.merge(df_HIV_subset[['country', 'year', 'hiv_cases']],
                           on=['country', 'year'], how='outer')

#Ordenar
df_merge.sort_values(['country', 'year'], inplace=True)

print(df_merge.head())


#Filtrar años entre 2010 al 2023
df_merge = df_merge[(df_merge['year'] >= 2010) & (df_merge['year'] <= 2023)]

# Verificar
print(df_merge.head(20))


In [ ]:
display(df_merge)

In [ ]:
display(df_)

In [ ]:
#merge del indice de pobreza y gastos en salud pib

